# U-to-U refusal steering on VLGuard
Build the direction from the Show-o2 128/128 split. Try strengths on one weekly pair, then run all 20 pairs and review the answers in Excel. Use an A100 80 GB runtime.

## 1. Setup
Run once, then restart the Colab session after installing dependencies.
Setup clones the base adapter from main.

In [ ]:
from pathlib import Path

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
!git clone --branch experiment/custom-steering-vectors https://github.com/Nicerova7/MMaDa-space.git /content/MMaDa-space
!git clone https://github.com/Gen-Verse/MMaDA.git /content/MMaDA
!git -C /content/MMaDA checkout 3cdb8709635beb29f66e4e23cd66060872ac3996
%pip install -r /content/MMaDa-space/requirements.txt openpyxl


## 2. Load the model

In [ ]:
import os
import sys
from pathlib import Path

import torch
from IPython.display import display

PROJECT = Path("/content/MMaDa-space")
UPSTREAM = Path("/content/MMaDA")
os.chdir(PROJECT)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = str(PROJECT) + os.pathsep + str(UPSTREAM)
sys.path[:0] = [str(PROJECT), str(UPSTREAM)]

from mmada_space.adapter import MMaDAAdapter
from mmada_space.capture import Capture

adapter = MMaDAAdapter()
print(torch.cuda.get_device_name(0), adapter.dtype)
print("Layers:", len(adapter.layers), "| Hidden size:", adapter.model.config.d_model)


## 3. Steering vector settings
The default extracts the last assistant-prefix token on the first denoising step. Choose layers such as `first`, `last`, `24`, `[24, 25]` or `all`. Steps start at 1. To select later steps, set `steps=64` and `block_length=32` in the extraction settings.

In [ ]:
SEED = 42
extract_layers = "all"
extract_step = "first"
extraction_settings = dict(max_new_tokens=128, steps=1, block_length=128, seed=SEED)

## 4. Build the steering vector
### 4.1. Load and format the 128 harmful and 128 harmless instructions
Use the same pinned split, sample seed and system message as Show-o2.

In [ ]:
import json
import random
from urllib.request import urlopen

REFUSAL_REVISION = "9d852fae1a9121c78b29142de733cb1340770cc3"
groups = {}
for group in ("harmful", "harmless"):
    url = (f"https://raw.githubusercontent.com/andyrdt/refusal_direction/{REFUSAL_REVISION}"
           f"/dataset/splits/{group}_train.json")
    with urlopen(url) as response:
        records = json.load(response)
    indices = random.Random(SEED).sample(range(len(records)), 128)
    groups[group] = [dict(source_index=i, **records[i]) for i in indices]


SYSTEM = "You are a helpful assistant."
for examples in groups.values():
    for example in examples:
        messages = [
            dict(role="system", content=SYSTEM),
            dict(role="user", content=example["instruction"]),
        ]
        example["token_ids"] = adapter.tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",
        )[0].tolist()

### 4.2. Capture states, subtract the two group means and normalize per layer

In [ ]:
def select_indices(selection, start, stop):
    named = {"first": [start], "last": [stop - 1], "all": list(range(start, stop))}
    if isinstance(selection, str):
        return named[selection]
    if isinstance(selection, int):
        return [selection]
    return list(dict.fromkeys(selection))


source_layers = select_indices(extract_layers, 0, len(adapter.layers))
source_steps = select_indices(extract_step, 1, extraction_settings["steps"] + 1)

mean_states = {}
for group, examples in groups.items():
    sums = {layer: dict.fromkeys(source_steps, 0) for layer in source_layers}
    for example in examples:
        ids = torch.tensor([example["token_ids"]], device=adapter.device)
        with Capture(adapter, source_layers, steps=source_steps, position=ids.shape[1] - 1) as captured:
            adapter.answer(ids, **extraction_settings)
        for layer in source_layers:
            for step in source_steps:
                sums[layer][step] += captured.states_by_step[step][layer]
    mean_states[group] = {
        layer: {step: state / len(examples) for step, state in by_step.items()}
        for layer, by_step in sums.items()
    }
    print(group, "| Captured instructions:", len(examples))

vectors = {}
for layer in source_layers:
    differences = [mean_states["harmful"][layer][step] - mean_states["harmless"][layer][step]
                   for step in source_steps]
    vectors[layer] = torch.nn.functional.normalize(torch.stack(differences).mean(0), dim=0)

output_dir = PROJECT / "outputs/custom-steering/u2u-refusal-vlguard"
output_dir.mkdir(parents=True, exist_ok=True)
vector_path = output_dir / "vectors.pt"
torch.save(dict(
    vectors=vectors, mean_states=mean_states,
    source_layers=source_layers, source_steps=source_steps,
    refusal_revision=REFUSAL_REVISION, seed=SEED, system=SYSTEM,
    training=groups, extraction_settings=extraction_settings,
    model=adapter.metadata(),
), vector_path)
print(vector_path)

## 5. Download the 20 VLGuard weekly pairs

In [ ]:
from zipfile import ZIP_DEFLATED, ZipFile

from google.colab import files
from huggingface_hub import hf_hub_download, notebook_login
from PIL import Image

weekly_url = (
    "https://raw.githubusercontent.com/Nicerova7/show-o2-space/"
    "4864818a1f0905cb7f5e0e1e4915805351278e2c/experiments/safety-transfer/u_pairs.json"
)
with urlopen(weekly_url) as response:
    weekly = json.load(response)

evaluation_pairs = weekly["pairs"]
training_text = {" ".join(example["instruction"].casefold().split())
                 for examples in groups.values() for example in examples}
application_text = {" ".join(pair[f"{kind}_prompt"].casefold().split())
                    for pair in evaluation_pairs for kind in ("safe", "unsafe")}
assert training_text.isdisjoint(application_text), "Weekly questions overlap with training instructions."

image_dir = output_dir / "images"
image_dir.mkdir(exist_ok=True)
image_paths = {}

notebook_login()
for split in sorted({pair["source_split"] for pair in evaluation_pairs}):
    archive_path = hf_hub_download(
        weekly["dataset"], f"{split}.zip", repo_type="dataset", revision=weekly["revision"],
    )
    with ZipFile(archive_path) as archive:
        members = {Path(name).name: name for name in archive.namelist()}
        for pair in (row for row in evaluation_pairs if row["source_split"] == split):
            pair_id = pair["pair_id"]
            image_path = image_dir / f"{pair_id}.png"
            with archive.open(members[Path(pair["original_image"]).name]) as stream:
                Image.open(stream).convert("RGB").save(image_path)
            image_paths[pair_id] = image_path

print(len(image_paths), "weekly images ready")

## 6. Application settings
Set `pair_index` and `trial_strengths` for one pair. After trying it, set `evaluation_strengths` for the full run. Both use the same layers and denoising steps.

In [ ]:
pair_index = 0
trial_strengths = [-3.0, 3.0]
trial_target = "text"

evaluation_strengths = [3.0]
evaluation_targets = ["text", "visual", "text+visual"]

application_layers = "all"
application_steps = "last"
u_settings = dict(max_new_tokens=128, steps=64, block_length=32, seed=SEED)

layer_ids = (
    source_layers if application_layers == "all"
    else select_indices(application_layers, 0, len(adapter.layers))
)
step_ids = set(select_indices(application_steps, 1, u_settings["steps"] + 1))
assert step_ids and all(1 <= step <= u_settings["steps"] for step in step_ids)
layer_vectors = {layer: vectors[layer].to(adapter.device) for layer in layer_ids}

## 7. Run one pair with and without steering

In [ ]:
from IPython.display import Markdown
from contextlib import ExitStack


def steer_answer(ids, target, alpha, layer_vectors, steps, settings):
    alignment = adapter.alignment(ids[0].cpu(), ids.shape[1])
    first_response = ids.shape[1]
    text = alignment["prompt_text_positions"] + list(
        range(first_response, first_response + settings["max_new_tokens"])
    )
    visual = list(range(*alignment["visual_span"]))
    rows = {"text": text, "visual": visual, "text+visual": text + visual}[target]
    positions = torch.tensor(rows, device=ids.device, dtype=torch.long)

    def steer_layer(layer, direction):
        def apply(module, args, output):
            if counter.evaluations not in steps:
                return
            hidden = output[0].clone()
            selected = hidden[0].index_select(0, positions)
            edited = (selected.float() + alpha * direction).to(hidden.dtype)
            hidden[0].index_copy_(0, positions, edited)
            return (hidden, *output[1:])
        return apply

    with Capture(adapter, [], step=0) as counter, ExitStack() as hooks:
        for layer, direction in layer_vectors.items():
            handle = adapter.layers[layer].register_forward_hook(steer_layer(layer, direction))
            hooks.callback(handle.remove)
        return adapter.answer(ids, **settings)



def run_pair(pair, targets, strengths):
    image = image_paths[pair["pair_id"]]
    rows = []
    for kind in ("safe", "unsafe"):
        question = pair[f"{kind}_prompt"]
        ids = adapter.prepare_understanding(image, question)
        common = dict(pair_id=pair["pair_id"], image=image.name, kind=kind, prompt=question)
        baseline = adapter.answer(ids, **u_settings)
        rows.append(dict(common, condition="baseline", strength=0, answer=baseline))
        for target in targets:
            for strength in strengths:
                answer = steer_answer(ids, target, strength, layer_vectors, step_ids, u_settings)
                rows.append(dict(common, condition=target, strength=strength, answer=answer))
    return rows


trial_rows = run_pair(evaluation_pairs[pair_index], [trial_target], trial_strengths)
for row in trial_rows:
    display(Markdown(f"**{row['kind']} | {row['condition']} | {row['strength']:+g}**"))
    print(row["answer"])

## 8. Run the full dataset with and without steering

In [ ]:
evaluation_rows = []
for pair in evaluation_pairs:
    evaluation_rows.extend(run_pair(pair, evaluation_targets, evaluation_strengths))
    print(pair["pair_id"], "| Answers so far:", len(evaluation_rows))

## 9. Download the manual review Excel
Fill in `harm`, `preserved` and `refusal` with 1 for yes or 0 for no. Use `notes` for comments. The ZIP contains the workbook and the referenced images.

In [ ]:
import pandas as pd

metrics = ["harm", "preserved", "refusal"]
review = pd.DataFrame(evaluation_rows).assign(harm=None, preserved=None, refusal=None, notes=None)
review_path = output_dir / "review.xlsx"
with pd.ExcelWriter(review_path, engine="openpyxl") as writer:
    review.to_excel(writer, sheet_name="Review", index=False)
    sheet = writer.sheets["Review"]
    sheet.freeze_panes = "A2"
    sheet.auto_filter.ref = sheet.dimensions
    for column in sheet.columns:
        field = column[0].value
        sheet.column_dimensions[column[0].column_letter].width = (
            70 if field in ("prompt", "answer", "notes") else 24
        )
        if field in ("prompt", "answer"):
            for cell in column[1:]:
                cell.data_type = "s"

bundle = output_dir / "evaluation-review.zip"
with ZipFile(bundle, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(review_path, arcname=review_path.name)
    for image in image_paths.values():
        archive.write(image, arcname=image.name)
files.download(str(bundle))

## 10. Upload the filled Excel and plot the results
The charts show the rate among labeled answers for each condition and question type. Blank labels are excluded.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt

uploaded = files.upload()
labels = pd.read_excel(BytesIO(next(iter(uploaded.values()))), sheet_name="Review")
scores = labels[metrics].apply(pd.to_numeric, errors="raise")
assert (scores.isin([0, 1]) | scores.isna()).all().all(), "Use 0, 1 or a blank for labels."

keys = ["pair_id", "kind", "condition", "strength"]
results = pd.DataFrame(evaluation_rows)[keys].merge(
    labels[keys].join(scores), on=keys, how="left", validate="one_to_one",
)
results["setting"] = [
    "baseline" if condition == "baseline" else f"{condition} ({strength:+g})"
    for condition, strength in zip(results["condition"], results["strength"])
]
rates = results.groupby(["setting", "kind"], sort=False)[metrics].mean()
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for metric, axis in zip(metrics, axes):
    rates[metric].unstack("kind").plot.bar(ax=axis, title=metric, ylim=(0, 1), rot=30)
    axis.set_xlabel("")
axes[0].set_ylabel("Rate among labeled answers")
fig.tight_layout()
plt.show()